<a href="https://colab.research.google.com/github/Rozieyati/Scalable-Multi-Model-Data-Pipeline-for-MovieLens-Analysis-Using-Spark-Cassandra-and-MongoDB/blob/main/Data_Management_27062026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Scalable Multi-Model Data Pipeline: Integrating Apache Spark, Cassandra, and MongoDB for MovieLens Analysis**

**Course:** Data Management (STQD6324)  
**Programme:** Master of Science (Data Science and Analytics)  
**Semester:** Semester 2, 2025/2026

## **1.0 Introduction**

The digital era has ushered in an unprecedented surge in user-generated interaction data, creating significant challenges for traditional data management architectures. The MovieLens 100k dataset acts as a critical benchmark for evaluating modern recommender systems; however, the velocity and volume of such data often exceed the processing capabilities of legacy Relational Database Management Systems (RDBMS).

This study investigates the deployment of a Scalable Multi-Model Data Pipeline designed to overcome these limitations through a polyglot persistence strategy. By integrating Apache Spark as the primary distributed processing engine, we enable high-performance parallel computation of large-scale interaction logs. To address the divergent requirements of analytical and operational workloads, the pipeline leverages two distinct NoSQL storage technologies:

*   **Apache Cassandra**, pecifically implemented via **DataStax Astra DB** for its cloud-native, partition-tolerant, and high-availability architecture optimised for aggregated analytical ranking, and

*   **MongoDB**, for its schema-flexible document-store capabilities, providing the agility necessary to manage evolving user demographic profiles.

This architecture is grounded in the principles of the **CAP Theorem** (Brewer, 2012), balancing consistency, availability, and partition tolerance. By effectively decoupling heavy analytical computations from user-centric profile management, this implementation provides a blueprint for resilient, industry-grade recommendation engines that remain highly performant as they scale.

## **2.0 Problem Statement**

Whilst traditional Relational Database Management Systems (RDBMS) have historically served as the bedrock for data storage, they are increasingly ill-equipped to handle the high-velocity, high-volume interaction logs characteristic of modern recommendation engines. The rigid, tabular structures of RDBMS impose significant computational bottlenecks, leading to latency issues when performing large-scale aggregations.

Furthermore, demographic data, which is essential for effective content personalisation, often lacks the structural uniformity required for traditional database modelling. This leads to information silos, where analytical insights are obscured, thereby hindering the development of resilient and data-driven user engagement strategies. As the user base expands, legacy architectures struggle to maintain the necessary throughput for real-time ranking and the flexibility required for evolving user-profile schemas.

Moreover, the overhead associated with managing physical infrastructure for these legacy systems limits the agility required by modern applications. Consequently, there is an urgent need to transition towards a distributed, multi-model approach that leverages cloud-native services like Astra DB and MongoDB. This transition decouples analytical processing from operational profile management, ensuring the system remains both scalable and performant in a production environment.


## **3.0 Research Objectives**

To address the limitations identified in the problem statement and to establish a robust framework for scalable data management, this study focuses on the following research objectives:


1.   **To engineer a distributed data pipeline** using **Apache Spark**, facilitating the efficient transformation and parallel processing of high-volume interaction logs from the MovieLens 100k dataset.

2. **To implement a polyglot persistence
architecture**, integrating **DataStax Astra DB** (as an implementation of **Apache Cassandra**) to ensure high-availability and low-latency access to aggregated movie rankings, and **MongoDB** to provide a flexible document-store for dynamic user demographic profiling.

3. **To perform multi-dimensional analysis** on user-content interactions, specifically focusing on rating centralisation, temporal drift, and demographic polarisation, to extract actionable behavioural insights.

4. **To validate the efficacy of the pipeline** through a comparative assessment of query performance, ensuring the system maintains resilience and consistency as per the requirements of the **CAP Theorem**.

5. **To demonstrate architectural scalability** by transitioning from legacy, siloed storage systems to an integrated, cloud-native framework that supports both real-time analytical queries and long-term data persistence.

## **4.0 Methodology**

This research adopts a distributed data lifecycle approach, leveraging a modular pipeline that ensures data reproducibility and structural integrity. The methodology is divided into four distinct phases, transitioning from raw data ingestion to polyglot persistence.

## **4.1 Data Acquisition and Preprocessing**

The ingestion phase is managed through a structured SparkSession to facilitate the loading of heterogeneous files into a unified analytical environment.

The ingestion process is configured to handle the distinct formats of the MovieLens dataset which includes tab separated data files and pipe separated user files. This ensures that the data maintains its structural integrity as it is ingested into the Spark environment.



In [72]:
# 1. Mount Google Drive (Run once)
from google.colab import drive
import os
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

drive.mount('/content/drive')

# 2. Define the exact path based on your folder structure
# Corrected to match: MyDrive/Data Management/ml-100k/
base_path = "/content/drive/MyDrive/Data Management/ml-100k/"

# 3. Verify path existence before initializing Spark
if not os.path.exists(base_path + "u.data"):
    print(f"ERROR: File not found at {base_path}u.data.")
    print("Available files in directory:", os.listdir(base_path) if os.path.exists(base_path) else "Directory does not exist")
else:
    print("Path verified successfully. Starting Spark ingestion...")

    # 4. Initialize SparkSession
    spark = SparkSession.builder \
        .appName("MovieLens_Ingestion") \
        .getOrCreate()

    # 5. Define schema for data integrity
    data_schema = StructType([
        StructField("user_id", IntegerType(), True),
        StructField("movie_id", IntegerType(), True),
        StructField("rating", IntegerType(), True),
        StructField("timestamp", IntegerType(), True)
    ])

    # 6. Perform Data Ingestion
    df_data = spark.read.csv(base_path + "u.data", sep='\t', schema=data_schema)
    df_users = spark.read.csv(base_path + "u.user", sep='|') \
        .toDF("user_id", "age", "gender", "occupation", "zip")

    # 7. Clean and Validate Data
    df_data = df_data.na.drop()
    df_users = df_users.na.drop()

    print("Data ingestion and cleaning successfully completed.")
    df_data.show(5)
    df_users.show(5)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Path verified successfully. Starting Spark ingestion...
Data ingestion and cleaning successfully completed.
+-------+--------+------+---------+
|user_id|movie_id|rating|timestamp|
+-------+--------+------+---------+
|    196|     242|     3|881250949|
|    186|     302|     3|891717742|
|     22|     377|     1|878887116|
|    244|      51|     2|880606923|
|    166|     346|     1|886397596|
+-------+--------+------+---------+
only showing top 5 rows
+-------+---+------+----------+-----+
|user_id|age|gender|occupation|  zip|
+-------+---+------+----------+-----+
|      1| 24|     M|technician|85711|
|      2| 53|     F|     other|94043|
|      3| 23|     M|    writer|32067|
|      4| 24|     M|technician|43537|
|      5| 33|     F|     other|15213|
+-------+---+------+----------+-----+
only showing top 5 rows


By defining a schema at the point of ingestion we enforce data typing which is a crucial step in maintaining consistency within a distributed environment. Unlike traditional relational database management systems where the schema is fixed at the storage level this schema on read approach in Apache Spark grants us the flexibility to transform data on the fly before persisting it into our chosen NoSQL storage systems. This step ensures that subsequent analytical operations such as calculating average ratings or filtering by demographic are performed on a high fidelity dataset. This significantly reduces the risk of runtime errors during the persistence phase and ensures that the data is ready for downstream analytical processing tasks.

## **4.2 Distributed Data Processing**

After the ingestion process, the raw data must be transformed into a format suitable for analytical tasks. Apache Spark facilitates this through the use of DataFrames, which provide a high level abstraction for handling structured data.

We perform necessary transformations to derive meaningful metrics. This includes grouping the interaction logs to calculate averages and joining the user profile data to allow for demographic filtering.



In [73]:
from pyspark.sql.functions import col, avg, count, desc

# Calculate average rating for each movie
df_movie_avg = df_data.groupBy("movie_id").agg(avg("rating").alias("avg_rating"))

# Identify top ten movies with the highest average ratings
top_ten_movies = df_movie_avg.orderBy(desc("avg_rating")).limit(10)

# Filter users who have rated at least fifty movies
active_user_ids = df_data.groupBy("user_id").count().filter(col("count") >= 50).select("user_id")

# Join with user profile data for demographic analysis
df_active_user_profiles = active_user_ids.join(df_users, "user_id")

print("Data processing and transformation tasks successfully executed.")

Data processing and transformation tasks successfully executed.


By defining a schema at the point of ingestion we enforce data typing which is a crucial step in maintaining consistency within a distributed environment. Unlike traditional relational database management systems where the schema is fixed at the storage level this schema on read approach in Apache Spark grants us the flexibility to transform data on the fly before persisting it into our chosen NoSQL storage systems. This step ensures that subsequent analytical operations such as calculating average ratings or filtering by demographic are performed on a high fidelity dataset. This significantly reduces the risk of runtime errors during the persistence phase and ensures that the data is ready for downstream analytical processing tasks.

## **5.0 Polyglot Persistence Implementation**

To fulfil the requirements of a resilient and cloud native data architecture, we have implemented a polyglot persistence strategy. This approach utilises specialised storage systems to accommodate the varying characteristics of our analytical and operational data whilst leveraging the agility of modern cloud services.

## **5.1 Apache Cassandra (Astra DB Persistence)**

We utilise DataStax Astra DB as our managed service for Apache Cassandra to store the aggregated movie rankings. Given that our analytical dashboard requires rapid read access to top rated movie lists, Astra DB provides the necessary high write throughput and fault tolerance required for such operations without the burden of infrastructure management.

In [74]:
from google.colab import userdata
from astrapy import DataAPIClient

# This fetches the value stored inside the secret named 'ASTRA_TOKEN'
token = userdata.get('ASTRA_TOKEN')
endpoint = userdata.get('ASTRA_ENDPOINT')

# Initialize the client using the variables
client = DataAPIClient(token)
db = client.get_database_by_api_endpoint(endpoint)

print("Successfully connected to Astra DB!")

Successfully connected to Astra DB!


The output of the data ingestion process serves as a verification of successful transformation and load operations. Upon execution, the system returns a collection of unique identifiers (inserted_ids) for each document migrated. These IDs confirm that the structured tabular data from the MovieLens dataset has been successfully mapped into a document-oriented format, preserving the integrity of the original records (user_id, movie_id, rating, and timestamp) within the Astra DB collection. The successful return of these identifiers validates the end-to-end connectivity between our transient Spark execution environment and our persistent NoSQL storage layer.

## **5.2 Retrieving Data from Astra DB**

The purpose of the retrieval phase is to validate the integrity and accessibility of the persisted data. In a Polyglot Persistence architecture, it is not enough to just store data; you must prove that the data stored in the NoSQL layer (Astra DB) remains accurate and is queryable with the speed expected by an analytical dashboard.

"The primary purpose of implementing read operations from Astra DB is to verify the efficacy of the persistence layer. By executing document-based queries, we ensure that the data ingested via Apache Spark maintains its schema integrity. Furthermore, this phase validates that the system can support the rapid, low-latency filtering required for real-time dashboard updates, confirming that the NoSQL choice is appropriate for the analytical requirements of the MovieLens dataset."

In [75]:
# Use the correct key 'avg_rating' found in your sample_doc
query = {"avg_rating": 5}
high_ratings = list(collection.find(query, limit=5))

if high_ratings:
    print(f"Top {len(high_ratings)} rated movies retrieved from NoSQL store:")
    for doc in high_ratings:
        print(f"Movie ID: {doc['movie_id']} | Rating: {doc['avg_rating']}")
else:
    print("No records found matching the criteria.")

Top 5 rated movies retrieved from NoSQL store:
Movie ID: 1653 | Rating: 5
Movie ID: 1189 | Rating: 5
Movie ID: 1500 | Rating: 5
Movie ID: 1536 | Rating: 5
Movie ID: 1201 | Rating: 5


The implementation of this pipeline highlighted the importance of decoupling compute and storage. By using Spark for batch processing and Astra DB for rapid read-access, we have built a system that is both computationally efficient and highly responsive. The successful retrieval of specific high-rated records validates that our data mapping strategy is effective and ready for integration into a user-facing analytical dashboard.

## **6.0 Exploratory Data Analysis (EDA)**

The primary purpose of Exploratory Data Analysis (EDA) in this pipeline is to perform descriptive analytics to transform raw, aggregated data into actionable insights. According to Kotu & Deshpande (2019), EDA is a fundamental prerequisite in data engineering, as it serves to identify data sparsity, assess quality, and detect potential biases that could negatively impact downstream machine learning performance.

By visualizing user demographics and rating distributions, we establish a robust empirical baseline. This phase bridges the gap between the technical storage layer (Astra DB) and the practical requirement of informed decision-making for recommendation systems.

## **6.2 Multivariate Data Visualization**

This section presents a comprehensive analytical suite designed to evaluate the integrity and behavioral trends of the processed MovieLens dataset. As supported by Sadalage & Fowler (2013), the utility of a persistent NoSQL architecture is maximized when paired with robust analytical visualization, allowing for the rapid identification of data patterns that are not immediately evident in tabular views.

## **6.1 Average Rating by Geographic Region.**

The purpose of this visualization is to analyze the variation in user sentiment across different geographic sectors. By calculating the average movie rating per region, we can identify "sentiment hotspots" where user satisfaction is exceptionally high and "cold spots" that may require investigation. This analysis is fundamental for understanding regional content preferences and serves as a diagnostic tool for identifying where the current content recommendation strategy may be underperforming or excelling.

In [80]:
import pandas as pd

# Load the datasets
# Ensure these files are uploaded to your Colab /content/ folder
df_data = pd.read_csv('/content/u.data', sep='\t', names=['user_id', 'movie_id', 'rating', 'timestamp'])
df_users = pd.read_csv('/content/u.user', sep='|', names=['user_id', 'age', 'gender', 'occupation', 'zip'])

# Merge datasets on user_id
df = pd.merge(df_data, df_users, on='user_id')

# Create the 'region' column using the first digit of the ZIP code
df['region'] = df['zip'].astype(str).str[0]
df = df[df['region'].str.isdigit()] # Retain only valid numeric regions

print("Data successfully prepared. DataFrame 'df' is ready.")

Data successfully prepared. DataFrame 'df' is ready.


In [86]:
import pandas as pd
import plotly.express as px

# 1. Map the ZIP code first-digit to Geographic Region Names
region_map = {
    '0': 'New England', '1': 'Northeast', '2': 'Mid-Atlantic',
    '3': 'South Atlantic', '4': 'Midwest', '5': 'Upper Midwest',
    '6': 'Central', '7': 'Southwest', '8': 'Mountain', '9': 'Pacific'
}

# 2. Process the data
# Extract the first digit from ZIP, then map to the region name
df['region'] = df['zip'].astype(str).str[0]
df['region_name'] = df['region'].map(region_map)

# 3. Aggregate data for the chart
chart_data = df.groupby('region_name')['rating'].mean().reset_index()

# 4. Generate the High-Intensity Chart
fig = px.bar(
    chart_data,
    x='region_name',
    y='rating',
    title="<b>Average Movie Rating by Geographic Sector</b>",
    color='rating',
    color_continuous_scale='Inferno',
    text='rating'
)

# 5. Styling
fig.update_traces(
    texttemplate='%{text:.2f}',
    textposition='outside',
    marker_line_width=1.5,
    marker_line_color="black"
)

fig.update_layout(
    title_x=0.5,
    title_font=dict(size=24, color='black', family='Arial Black'),
    xaxis_title="Geographic Sector",
    yaxis_title="Mean Rating (1-5 Scale)",
    yaxis=dict(range=[0, 5.5]),
    plot_bgcolor='white',
    paper_bgcolor='white'
)

fig.show()

The visualization of average ratings across geographic sectors reveals a significant sentiment heterogeneity, which suggests that user satisfaction is deeply influenced by regional cultural contexts rather than universal preference. By identifying these distinct sentiment clusters, the platform can transition from a generic global averaging model to a geographically tailored recommendation strategy, thereby mitigating the risk of content market mismatch and increasing long-term user retention. As noted by Kotu and Deshpande (2019), geographic segmentation is a fundamental prerequisite in recommendation systems to avoid the homogeneity trap because failure to account for spatial behavioral variance significantly diminishes the precision of personalized content delivery.

This localized approach is further supported by Aggarwal (2016), who emphasizes that granularity in user feature processing, specifically identifying regional consumption patterns, is critical to overcoming popularity bias and addressing the cold start problem for new users. By leveraging these insights, developers can optimize content inventory distribution to better align with the unique cultural preferences of specific regions, ensuring that the platform recommendation architecture remains context aware and empirically robust.

## **6.2 Geographic Hierarchy and Engagement Distribution**

The Sunburst visualization provides a clear window into the structural distribution of user engagement across the US. By centralizing the data flow, we can instantly perceive which geographic sectors dominate the platform’s activity, revealing an asymmetrical distribution that suggests high regional concentration. This hierarchical view is critical for identifying power regions—areas where user activity is dense enough to sustain highly personalized recommendation models.

As noted by Few (2012), when data is organized hierarchically, it reduces the cognitive load on stakeholders, allowing them to instantly differentiate between dominant and emerging markets. Furthermore, this approach aligns with the principles set forth by Aggarwal (2016), where granular segment identification is a mandatory step in reducing popularity bias and optimizing resource allocation within recommendation engines. Addressing the disparities highlighted here enables a more proactive, data informed strategy for content distribution and targeted infrastructure scaling.

In [88]:
import plotly.express as px

# 1. Define the mapping
region_map = {
    '0': 'New England', '1': 'Northeast', '2': 'Mid-Atlantic',
    '3': 'South Atlantic', '4': 'Midwest', '5': 'Upper Midwest',
    '6': 'Central', '7': 'Southwest', '8': 'Mountain', '9': 'Pacific'
}

# 2. Map and drop unmatched data
# We map the first digit. Any digit not in our map will become NaN (Not a Number)
df['region_name'] = df['zip'].astype(str).str[0].map(region_map)

# Remove all rows where 'region_name' is missing (i.e., not found in our map)
df = df.dropna(subset=['region_name'])

# 3. Prepare the data for the Sunburst
df_sun = df.groupby(['region_name']).size().reset_index(name='count')
df_sun['Global'] = 'US Regions'

# 4. Generate the Sunburst Chart
fig = px.sunburst(
    df_sun,
    path=['Global', 'region_name'],
    values='count',
    color='count',
    color_continuous_scale='Inferno',
    title="<b>Geographic Distribution of User Engagement</b>"
)

# 5. Styling
fig.update_layout(
    title_x=0.5,
    title_font=dict(size=24, color='black', family='Arial Black'),
    margin=dict(t=80, l=0, r=0, b=0)
)
fig.update_traces(textinfo="label+value")

fig.show()

## **6.3 Gender Demographic Distribution**

The purpose of this visualization is to analyze the gender composition of the platform’s user base. Understanding this demographic breakdown is essential for evaluating potential algorithmic bias, as it ensures that the recommendation engine remains inclusive and does not inadvertently favor the preferences of a single demographic group over another.




In [90]:
import plotly.graph_objects as go

# 1. Get counts and map full names
# This assumes your 'gender' column has 'M' and 'F'
gender_counts = df['gender'].value_counts()
full_name_map = {'M': 'Male', 'F': 'Female'}

# Create a clean dictionary for the chart
labels = [full_name_map.get(g, g) for g in gender_counts.index]
values = gender_counts.values.tolist()

# 2. Define high-contrast colors
colors = ['#1a2a6c', '#fdbb2d'] # Navy for Male, Gold for Female

# 3. Initialize donut chart
fig = go.Figure(data=[go.Pie(
    labels=labels,
    values=values,
    hole=0.6,
    marker=dict(
        colors=colors,
        line=dict(color='#ffffff', width=5)
    ),
    textinfo='label+percent',
    textfont=dict(size=18, family='Arial Black'),
    sort=False
)])

# 4. Update layout
fig.update_layout(
    title=dict(
        text="<b>Gender Demographic Distribution</b>",
        x=0.5,
        font=dict(size=25, family="Arial Black", color='#333333')
    ),
    annotations=[dict(
        text='GENDER',
        x=0.5, y=0.5,
        font_size=30,
        showarrow=False,
        font_family="Arial Black",
        font_color='#333333'
    )],
    showlegend=False,
    paper_bgcolor='white',
    plot_bgcolor='white'
)

fig.show()

The observed gender demographic skew, featuring approximately 71% male to 29% female representation, serves as a critical indicator for market segmentation and platform strategy. This asymmetry, as documented by Aggarwal (2016), inherently risks popularity bias within recommendation engines, which tend to optimize for the preferences of the majority group while inadvertently marginalizing minority segments. Consequently, Crawford (2021) warns that such skews in training data often encode and amplify societal inequities, transforming them into algorithmic outputs that favor one demographic over another.

To mitigate these risks and foster an inclusive ecosystem, platforms must shift toward fairness aware machine learning architectures, a necessity emphasized by Barocas, Hardt, and Narayanan (2019), who argue that consistent demographic auditing is essential to maintaining platform health. Furthermore, Noble (2018) provides the essential social framework for this analysis, illustrating how systems that fail to account for such diversity risk creating consumption silos that diminish long-term user retention.  By acknowledging these academic perspectives, it becomes clear that the current demographic imbalance requires proactive, fairness driven calibration to ensure equitable content delivery for all user segments.

## **6.3 Rating Variance by Age**

The third visualisation is a scatter plot that maps rating variance against user age. While mean ratings provide a central tendency, variance reveals the consistency of user behaviour. This analysis is vital for identifying whether rating behaviour becomes more stable or volatile as users transition through different life stages.

We calculate the variance of ratings per user and merge this with the user demographic metadata. This allows us to observe if there is a statistical correlation between age and the tendency to provide extreme ratings (e.g., only 1s or 5s) versus moderate, consistent ratings.

In [93]:
import plotly.express as px

# Histogram for age frequency
fig = px.histogram(df,
                   x='age',
                   nbins=30,
                   title='<b>User Age Distribution</b>',
                   labels={'age': 'Age'}, # Changed label to 'Age' for conciseness
                   template='plotly_white')

# Adding a modern gradient look
fig.update_traces(
    marker_color='#6C63FF',      # Modern Electric Purple
    marker_line_color='white',   # Clean white borders
    marker_line_width=1.0,
    opacity=0.85                 # Soft transparency for a premium feel
)

# Adding median line for analytical depth
median_age = df['age'].median()
fig.add_vline(x=median_age, line_dash="solid", line_color="#FF6584",
              annotation_text=f"Median: {median_age}",
              line_width=2)

fig.update_layout(
    title_x=0.5,
    title_font=dict(size=22, family="Arial", color='#333333'),
    xaxis_title='Age',
    yaxis_title='Number of Users',
    bargap=0.05 # Smaller gap for a continuous, flowing look
)

fig.show()

The identification of a median age of 30 serves as a critical strategic anchor for the platform, defining the core demographic around which engagement strategies must be centered. This cohort, situated in the prime of professional and digital maturity, represents the most stable segment of your user base. According to Kahneman (2011), life stage heuristics significantly influence decision-making processes, implying that content recommendations calibrated toward the preferences of this median group will yield higher retention and satisfaction rates. By prioritizing content alignment with the established consumption habits of 30-year-old users, the platform can effectively stabilize its primary revenue streams and engagement metrics.

However, reliance on this median baseline necessitates a robust commitment to algorithmic neutrality to prevent the formation of exclusionary consumption silos. As emphasized by Barocas, Hardt, and Narayanan (2019), systems optimized solely for a central demographic risk reinforcing popularity bias, which may systematically marginalize the preferences of younger or older minority segments. To foster a sustainable and equitable ecosystem, it is essential to utilize the median age not as a restrictive target, but as a benchmark for demographic auditing. Implementing fairness-aware architectures that deliberately balance content exposure across diverse age cohorts will ensure that the platform remains inclusive and adaptable to long-term market shifts, ultimately enhancing the overall lifetime value of the entire user base.

## **6.4 Occupation Distribution Analysis**

The primary purpose of this visualization is demographic segmentation. By visualizing user counts per occupation.

In [100]:
import plotly.express as px

# 1. Clean the Data: Filter, Rename, and Capitalize
# Create a copy to avoid SettingWithCopy warnings
df_occ = df.copy()

# Remove 'unknown' and handle 'none'
df_occ = df_occ[df_occ['occupation'].str.lower() != 'unknown']
df_occ['occupation'] = df_occ['occupation'].replace(['none', 'None'], 'Unemployed')

# Capitalize all remaining occupations
df_occ['occupation'] = df_occ['occupation'].str.capitalize()

# 2. Prepare Data
occ_counts = df_occ['occupation'].value_counts().reset_index()
occ_counts.columns = ['Occupation', 'Count']

# 3. Generate Horizontal Bar Chart with Log Scale
fig = px.bar(
    occ_counts,
    x='Count',
    y='Occupation',
    orientation='h',
    title='<b>User Occupation Distribution</b>',
    color='Count',
    color_continuous_scale='Bluered',
    text='Count'
)

# 4. Professional Styling & Log Axis
fig.update_layout(
    title_x=0.5,
    title_font=dict(size=22, family="Arial Black", color='#333333'),
    xaxis_title='Number of Users (Log Scale)',
    yaxis_title='Occupation Category', # Y-axis name added here
    xaxis_type="log",
    plot_bgcolor='white',
    paper_bgcolor='white'
)

# Reverse axis so highest counts appear at the top
fig.update_yaxes(autorange="reversed")

# Ensure labels are clear
fig.update_traces(texttemplate='%{text}', textposition='outside')

fig.show()

The horizontal bar chart serves as a strategic demographic segmentation tool, enabling stakeholders to identify the primary professional pillars within the user base. By employing a logarithmic scale to manage the wide variance in group sizes, ranging from high frequency categories like Student to niche groups like Retired, this visualization prevents dominant segments from visually suppressing the long tail data. This methodology is consistent with Edward Tufte’s principles of graphical excellence, which emphasize that data visualizations must maximize the data ink ratio while providing clarity and integrity, ensuring that even the smallest demographic segments are represented accurately for comparative analysis.

From an analytical standpoint, this profile reveals specific skewness in user engagement that directly informs content strategy. Aligning with data driven marketing frameworks, recognizing the platform’s core users, such as technical or academic professionals, allows for highly targeted recommendation algorithms and personalized content delivery. Furthermore, the rigorous data cleaning applied, including the standardization of labels via capitalization and the reclassification of none to Unemployed, adheres to best practices in data preprocessing as outlined in Python for Data Analysis (McKinney, 2017), ensuring that the resulting insights are robust, unambiguous, and suitable for high level executive decision making.

In [116]:
import pgeocode
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# 1. Aggregate and process data
df_location_pandas = df.groupby('zip')['rating'].count().reset_index()
df_location_pandas.columns = ['zip_code', 'total_views']
df_location_pandas['zip_code'] = df_location_pandas['zip_code'].astype(str).str.strip().str.zfill(5)

# 2. Geocode
nomi = pgeocode.Nominatim('us')
geo_data = nomi.query_postal_code(df_location_pandas['zip_code'].tolist())
df_location_pandas['state'] = geo_data['state_code'].fillna('Unknown')

# 3. Identify Top 5 States
df_state = df_location_pandas.groupby('state', as_index=False)['total_views'].sum()
df_state = df_state[df_state['state'] != 'Unknown']
top_5_states = df_state.nlargest(5, 'total_views')['state'].tolist()

# 4. Categorize for Highlighting
df_state['Status'] = df_state['state'].apply(lambda x: 'Top 5' if x in top_5_states else 'Others')

# 5. Generate Map
fig = px.choropleth(
    df_state,
    locations="state",
    locationmode="USA-states",
    color="Status",
    scope="usa",
    color_discrete_map={'Top 5': '#ff7f0e', 'Others': '#d3d3d3'}, # Vibrant orange for Top 5, Gray for others
    labels={'Status': 'Engagement Level'}
)

# 6. Add state labels
fig.add_trace(
    go.Scattergeo(
        locations=df_state['state'],
        locationmode="USA-states",
        text=df_state['state'],
        mode="text",
        textfont=dict(size=10, color='black')
    )
)

# 7. Apply layout
fig.update_layout(
    title={'text': '<b>Top 5 Regions by Viewer Engagement</b>', 'x': 0.5, 'xanchor': 'center'},
    geo=dict(showlakes=True, lakecolor='rgb(255, 255, 255)'),
    margin={"r": 0, "t": 50, "l": 0, "b": 0}
)

fig.show()

In [104]:
import pgeocode
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# 1. Aggregate Data
df_location_pandas = df.groupby('zip')['rating'].count().reset_index()
df_location_pandas.columns = ['zip_code', 'total_views']

# 2. Clean and Geocode
df_location_pandas['zip_code'] = df_location_pandas['zip_code'].astype(str).str.strip().str.zfill(5)
nomi = pgeocode.Nominatim('us')
geo_data = nomi.query_postal_code(df_location_pandas['zip_code'].tolist())
df_location_pandas['state'] = geo_data['state_code'].fillna('Unknown')

# 3. Aggregate by state and filter
df_state = df_location_pandas.groupby('state', as_index=False)['total_views'].sum()
df_state = df_state[df_state['state'] != 'Unknown']

# 4. Generate Choropleth Map
fig = px.choropleth(
    df_state,
    locations="state",
    locationmode="USA-states",
    color="total_views",
    scope="usa",
    color_continuous_scale="Reds",
    labels={'state': 'State', 'total_views': 'Total Views'}
)

# 5. Add state text labels
fig.add_trace(
    go.Scattergeo(
        locations=df_state['state'],
        locationmode="USA-states",
        text=df_state['state'],
        mode="text",
        textfont=dict(size=10, color='black'),
        showlegend=False
    )
)

# 6. Apply Centered Title and Final Layout
fig.update_layout(
    title={
        'text': '<b>Top Movie Viewers Breakdown by US States</b>',
        'x': 0.5,
        'xanchor': 'center'
    },
    geo=dict(showlakes=True, lakecolor='rgb(255, 255, 255)'),
    margin={"r": 0, "t": 50, "l": 0, "b": 0}
)

fig.show()

In [114]:
import pgeocode
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# 1. Aggregate and process data
df_location_pandas = df.groupby('zip')['rating'].count().reset_index()
df_location_pandas.columns = ['zip_code', 'total_views']
df_location_pandas['zip_code'] = df_location_pandas['zip_code'].astype(str).str.strip().str.zfill(5)

# 2. Geocode to get State Codes
nomi = pgeocode.Nominatim('us')
geo_data = nomi.query_postal_code(df_location_pandas['zip_code'].tolist())
df_location_pandas['state'] = geo_data['state_code'].fillna('Unknown')

# 3. Create Full State Name Mapping
us_state_abbrev = {
    'AL': 'Alabama', 'AK': 'Alaska', 'AZ': 'Arizona', 'AR': 'Arkansas', 'CA': 'California',
    'CO': 'Colorado', 'CT': 'Connecticut', 'DE': 'Delaware', 'FL': 'Florida', 'GA': 'Georgia',
    'HI': 'Hawaii', 'ID': 'Idaho', 'IL': 'Illinois', 'IN': 'Indiana', 'IA': 'Iowa',
    'KS': 'Kansas', 'KY': 'Kentucky', 'LA': 'Louisiana', 'ME': 'Maine', 'MD': 'Maryland',
    'MA': 'Massachusetts', 'MI': 'Michigan', 'MN': 'Minnesota', 'MS': 'Mississippi', 'MO': 'Missouri',
    'MT': 'Montana', 'NE': 'Nebraska', 'NV': 'Nevada', 'NH': 'New Hampshire', 'NJ': 'New Jersey',
    'NM': 'New Mexico', 'NY': 'New York', 'NC': 'North Carolina', 'ND': 'North Dakota', 'OH': 'Ohio',
    'OK': 'Oklahoma', 'OR': 'Oregon', 'PA': 'Pennsylvania', 'RI': 'Rhode Island', 'SC': 'South Carolina',
    'SD': 'South Dakota', 'TN': 'Tennessee', 'TX': 'Texas', 'UT': 'Utah', 'VT': 'Vermont',
    'VA': 'Virginia', 'WA': 'Washington', 'WV': 'West Virginia', 'WI': 'Wisconsin', 'WY': 'Wyoming'
}

# 4. Final aggregation
df_state = df_location_pandas.groupby('state', as_index=False)['total_views'].sum()
df_state = df_state[df_state['state'] != 'Unknown']
df_state['state_full'] = df_state['state'].map(us_state_abbrev).fillna(df_state['state'])

# 5. Identify Bottom 5
bottom_5_list = df_state.nsmallest(5, 'total_views')['state'].tolist()
df_state['Engagement'] = df_state['state'].apply(
    lambda x: 'Bottom 5' if x in bottom_5_list else 'Others'
)

# 6. Generate Interactive Map
fig = px.choropleth(
    df_state,
    locations="state",
    locationmode="USA-states",
    color="Engagement",
    scope="usa",
    color_discrete_map={'Bottom 5': '#e41a1c', 'Others': '#cccccc'},
    hover_name="state_full",
    hover_data={'total_views': True, 'Engagement': False, 'state': False},
    labels={'total_views': 'Total View Count'}
)

# 7. Add Labels Directly to the Map
fig.add_trace(
    go.Scattergeo(
        locations=df_state['state'],
        locationmode="USA-states",
        text=df_state['state'],
        mode="text",
        textfont=dict(size=10, color='black')
    )
)

# 8. Layout and Instructions
fig.update_layout(
    title={
        'text': '<b>Bottom 5 Regions by Engagement (Hover for Details)</b>',
        'x': 0.5,
        'xanchor': 'center'
    },
    geo=dict(showlakes=True, lakecolor='rgb(255, 255, 255)'),
    margin={"r":0, "t":50, "l":0, "b":0}
)

fig.show()

In [ ]:
# 6. Generate Interactive Map with Explicit Hover Data
fig = px.choropleth(
    df_state,
    locations="state",
    locationmode="USA-states",
    color="Engagement",
    scope="usa",
    color_discrete_map={'Bottom 5': '#e41a1c', 'Others': '#cccccc'},
    hover_name="state_full", # This sets the title of the hover box
    hover_data={
        'total_views': True,   # This ensures the View count is visible
        'Engagement': False,   # Hides the redundant status label
        'state': False         # Hides the abbreviation
    },
    labels={'total_views': 'Total View Count'}
)

## **5.2.7 Summary Statistics**

Descriptive statistics were generated to provide an overview of the dataset and to identify potential outliers, extreme values, and general distribution patterns before conducting further analysis.

According to Tukey (1977), exploratory descriptive analysis is an essential step in understanding the characteristics of a dataset prior to applying advanced statistical or machine learning techniques.

In [ ]:
# GENERATE SUMMARY STATISTICS

df.describe(include='all')

The descriptive statistics provide an overview of the selected and cleaned dataset used for analysis. The results show that the dataset contains 2,561 disaster records across 190 countries, 5 regions, and 12 disaster types between 2020 and 2025. The most frequently recorded country is the United States of America, while Asia represents the most frequent region and Flood is the most common disaster type.

For numerical variables, the results indicate a highly right-skewed distribution of disaster impacts. The median value for Total Deaths is 4, while the maximum value reaches 53,000, suggesting that most disaster events result in low fatalities, but a small number of extreme events contribute disproportionately high impacts. A similar pattern is observed for Total Affected and Total Damage ('000 US$), where extreme values significantly exceed the central tendency measures.

This indicates the presence of heavy-tailed distributions and extreme outliers, which is a common characteristic in disaster datasets due to rare but high-impact catastrophic events. Such variability suggests that simple descriptive measures alone are insufficient to fully understand underlying patterns.

Therefore, further exploratory data analysis and visualisation are required to better examine distribution patterns, while advanced techniques such as clustering may be useful to identify hidden structures within disaster severity levels.

# **6.0 Results and Discussion**
Exploratory Data Analysis (EDA) was conducted to identify patterns, trends, and relationships within the disaster dataset. Data visualisation techniques were applied to enhance the understanding of disaster frequency, geographical distribution, human impacts, and economic losses.

According to Tukey (1977), exploratory data analysis is a fundamental step in identifying meaningful patterns and structures in data before applying advanced statistical or machine learning techniques.

In this section, the cleaned dataset is analysed using multiple visualisation techniques, including trend analysis, bar charts, heatmaps, scatter plots, correlation analysis, and clustering visualisation. Each visualisation is accompanied by interpretation to explain the key patterns and insights derived from the data.

## **6.1 Evolution of Disaster Frequency and Types**

This stacked area chart presents the annual evolution of natural disaster events by disaster type between 2020 and 2025. The purpose of this visualisation is to examine changes in both the frequency and composition of disaster types over time at a global scale.

This analysis directly supports Research Objective 1, which aims to identify the most frequent natural disaster types worldwide between 2020 and 2025. It also addresses the related research questions on disaster frequency distribution and temporal changes in disaster occurrence throughout the study period.

By visualising trends across multiple disaster categories, this section enables a clearer understanding of how different disaster types contribute to overall global disaster patterns over time.

In [ ]:
# 6.1 EVOLUTION OF DISASTER FREQUENCY AND TYPES

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Prepare data
pivot_df = (
    df.groupby(["Start Year", "Disaster Type"])
      .size()
      .unstack(fill_value=0)
)

# Sort disaster types by total frequency
pivot_df = pivot_df[pivot_df.sum().sort_values(ascending=False).index]

# Fire-inspired color palette
fire_colors = [
    "#FFE66D",  # warm yellow
    "#FFB703",  # gold
    "#FB8500",  # orange
    "#FF6B35",  # fiery orange-red
    "#E63946",  # strong red
    "#D00000",  # deep red
    "#9D0208",  # dark crimson
    "#6A040F",  # maroon
    "#F77F00",  # amber orange
    "#FAA307",  # bright amber
    "#DC2F02",  # hot red-orange
    "#FF5400"   # lava orange
]

# Match number of colors with number of disaster types
colors = fire_colors[:len(pivot_df.columns)]

# Dark dramatic theme
fig, ax = plt.subplots(figsize=(16, 8), facecolor="#0d0d0d")
ax.set_facecolor("#111111")

# Plot stacked area chart
pivot_df.plot.area(
    ax=ax,
    stacked=True,
    alpha=0.95,
    color=colors,
    linewidth=1.2
)

# Titles and labels
ax.set_title(
    "Evolution of Disaster Frequency and Types (2020–2025)",
    fontsize=20,
    fontweight="bold",
    color="#FFD166",
    pad=20
)

ax.set_xlabel("Year", fontsize=13, color="white", labelpad=10)
ax.set_ylabel("Number of Disaster Events", fontsize=13, color="white", labelpad=10)

# Ticks
ax.tick_params(axis='x', colors='white', labelsize=11)
ax.tick_params(axis='y', colors='white', labelsize=11)

# Grid
ax.grid(axis="y", linestyle="--", alpha=0.18, color="white")

# Remove top/right spines and style others
for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)

ax.spines["left"].set_color("white")
ax.spines["bottom"].set_color("white")

# Legend
legend = ax.legend(
    title="Disaster Type",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    frameon=True,
    facecolor="#1a1a1a",
    edgecolor="white",
    fontsize=10,
    title_fontsize=11
)

plt.setp(legend.get_texts(), color="white")
plt.setp(legend.get_title(), color="#FFD166")

plt.tight_layout()
plt.show()

The stacked area chart shows the annual distribution of natural disaster events by disaster type between 2020 and 2025. The visualization indicates that floods and storms consistently dominate disaster occurrences across the study period, making them the most frequent disaster categories in the dataset.

This pattern is consistent with global disaster reporting in the EM-DAT database (CRED), which identifies hydrometeorological hazards as the most common disaster types worldwide. Furthermore, the Intergovernmental Panel on Climate Change (IPCC, 2023) reports that climate change has increased the frequency and intensity of extreme weather events, which explains the dominance of flood and storm-related disasters.

In contrast, geophysical disasters such as earthquakes occur less frequently but remain relatively stable over time, indicating that they are less influenced by short-term climate variability.

Overall, the results suggest that disaster occurrence is highly concentrated in climate-related hazards rather than being evenly distributed across all disaster types. This finding supports further analysis on regional distribution and impact severity to better understand global disaster risk patterns.

## **6.2 Distribution of Disaster Types**

This bar chart illustrates the distribution of natural disaster types recorded globally between 2020 and 2025. The purpose of this visualisation is to identify which disaster categories occur most frequently within the dataset.

Understanding the distribution of disaster types is important for identifying dominant hazard patterns and supporting disaster risk prioritisation at the global level.

The bar chart illustrates a highly skewed distribution of disaster types recorded between 2020 and 2025, where floods and storms dominate global disaster occurrences. Floods represent the highest number of events (1,025), followed by storms (803), indicating a strong concentration of hydrometeorological hazards in the dataset.

The use of a logarithmic scale is necessary due to the extreme disparity between high-frequency disaster types (floods and storms) and low-frequency events such as mass movement (dry) and glacial lake outburst floods. This transformation allows clearer visualisation of both dominant and rare disaster categories within the same chart.

This uneven distribution is consistent with findings from UNDRR (2022), which reports that hydrometeorological hazards account for the majority of globally recorded disaster events. Similarly, EM-DAT data confirms that climate-related disasters are significantly more frequent than geophysical hazards.

According to Tukey (1977), real-world datasets often exhibit highly skewed distributions, where a small number of categories dominate while a long tail of rare events exists. This pattern is clearly observed in the current dataset.

Overall, the results indicate that disaster risk is heavily concentrated in a small number of high-frequency hazard types. This highlights the importance of prioritising flood and storm mitigation strategies in global disaster risk reduction planning, while still maintaining preparedness for low-frequency but high-impact disaster events.

## **6.3 Disaster Distribution Across Regions and Years**

This heatmap visualises the distribution of disaster events across different regions and years between 2020 and 2025. The purpose of this analysis is to identify spatial and temporal patterns in global disaster occurrences.

By combining regional and yearly dimensions, this visualisation provides insights into whether certain regions experience consistently higher disaster frequencies over time.

The heatmap presents the normalized distribution of disaster events across regions and years between 2020 and 2025, allowing proportional comparison across regions.

The results show clear spatiotemporal variation in disaster occurrence patterns. Europe records the highest proportional peak in 2023 (0.27), indicating a significant concentration of disaster events during that year compared to other regions. Oceania shows relatively higher proportions in the earlier years, particularly 2020 and 2021, before declining in later years. In contrast, Asia and Africa demonstrate relatively more stable distributions across the entire period, indicating consistent exposure to disaster events over time. The Americas show moderate variation without extreme peaks, suggesting a more balanced distribution of disaster occurrences across the study period.

Overall, the findings confirm that disaster risk is highly heterogeneous across both spatial and temporal dimensions. Certain regions experience concentrated peaks in specific years, while others maintain more stable patterns. This highlights the importance of region-specific disaster risk management strategies and continuous monitoring of temporal fluctuations in disaster occurrence.

## **6.4 Disaster Risk Landscape: Human Impact vs Economic Loss**

This section visualises the relationship between human impact and economic loss across different disaster types between 2020 and 2025. The analysis is conducted using aggregated values of total affected population and total economic damage.

The purpose of this visualisation is to identify disaster categories that simultaneously produce high human and economic impacts, providing a more comprehensive understanding of disaster severity patterns.

The scatter plot illustrates the relationship between human impact (total affected population) and economic losses across different disaster types between 2020 and 2025. Both variables are transformed using a logarithmic scale to reduce skewness caused by extreme values and to improve comparability across disaster categories.

The results show a clear heterogeneous distribution of disaster types, with no strong linear relationship between human impact and economic loss. Instead, disaster types form distinct patterns across the plot. Floods, storms, earthquakes, and droughts are positioned in the upper-right region, indicating that these disasters are associated with both high human impact and high economic losses, representing the highest severity group.

In contrast, infestations and mass movement (dry) are located in the lower-left region, reflecting consistently low human and economic impacts. Epidemics display a distinct pattern, where human impact is relatively high compared to economic losses, suggesting that their effects are more population-driven rather than economically destructive.

Overall, the results confirm that disaster impacts are multidimensional and non-linear, with different disaster types exhibiting different impact profiles. This highlights the importance of using combined human and economic indicators to better understand disaster severity and supports the use of clustering-based approaches for identifying risk categories.

## **6.5 Correlation Analysis of Disaster Impact Variables**

This section examines the relationships between key numerical variables in the disaster dataset, including Total Deaths, No. Injured, Total Affected, and Total Economic Damage.

The purpose of this analysis is to determine whether strong statistical relationships exist between human impact indicators and economic losses, and whether disasters with high human impacts are also associated with higher financial costs.

The correlation analysis provides insights into the underlying structure of the dataset by identifying the degree of association between variables. Strong positive relationships would indicate that disasters affecting large populations also tend to generate higher economic losses, while weak correlations suggest that human and economic impacts may occur independently depending on disaster type.

Understanding these relationships is important for supporting further machine learning analysis, particularly clustering, as correlation patterns help justify whether impact variables can meaningfully group disaster events into distinct severity categories.

The correlation matrix presents the relationships between key disaster impact variables, including Total Deaths, No. Injured, Total Affected, and Total Economic Damage between 2020 and 2025.

The results indicate generally weak positive correlations among most variables. The strongest relationship is observed between Total Deaths and No. Injured (r = 0.30), which represents a weak-to-moderate positive correlation. This suggests that disasters causing higher fatalities tend to be associated with increased injuries, reflecting the direct human impact of severe disaster events.

However, correlations between human impact variables and economic damage are very weak (ranging from 0.06 to 0.16). This indicates that economic losses are not strongly dependent on human impact levels. In other words, disasters affecting large populations do not necessarily result in high financial damage, and vice versa.

This weak correlation structure suggests that disaster impacts are multidimensional and influenced by different underlying factors such as infrastructure resilience, economic development, population density, and hazard type. It also indicates that human and economic impacts behave differently within the dataset.

Overall, the findings confirm that disaster impact variables are only weakly interrelated. This justifies the use of multivariate techniques such as clustering analysis to better identify hidden patterns and groupings within disaster severity profiles.

## **7.0 K-Means Clustering Analysis**

This section applies K-Means clustering, an unsupervised machine learning technique, to group disaster events based on their human and economic impact characteristics.

Unlike supervised learning, K-Means does not use predefined labels. Instead, it identifies natural groupings within the dataset by minimizing the distance between data points and cluster centroids through iterative optimization (MacQueen, 1967; Lloyd, 1982).

In this study, clustering is performed using four key variables: Total Deaths, No. Injured, Total Affected, and Total Economic Damage ('000 US$), which represent the severity of disaster impacts.

The purpose of this analysis is to classify disaster events into distinct vulnerability groups, which can support better understanding of disaster severity patterns and improve risk-based decision-making. This approach is widely used in data mining applications for pattern discovery in complex datasets (Jain, 2010; Han, Kamber & Pei, 2012).

## **7.1 Data Preparation**

Before applying K-Means clustering, the dataset is standardised to ensure that all variables contribute equally to the analysis. This is necessary because K-Means is a distance-based algorithm that is sensitive to differences in scale between variables such as population counts and economic values.

In this study, feature scaling is applied to normalise the range of numerical variables so that no single variable dominates the clustering process due to its magnitude. Standardisation is commonly used in clustering analysis to improve model stability and ensure meaningful distance calculations (Jain, 2010; Han, Kamber & Pei, 2012).

This preprocessing step ensures that all variables contribute equally to the Euclidean distance computation used in K-Means clustering, resulting in more accurate and reliable cluster formation.

In [ ]:
# DATA PREPARATION

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# Select numerical features
cluster_data = df[[
    "Total Deaths",
    "No. Injured",
    "Total Affected",
    "Total Damage ('000 US$)"
]]

# Standardisation (IMPORTANT for KMeans)
scaler = StandardScaler()
scaled_data = scaler.fit_transform(cluster_data)

The standardisation process ensures that all variables are transformed to a comparable scale with mean zero and unit variance. This prevents variables with larger numerical ranges, such as Total Affected or Total Economic Damage, from disproportionately influencing the clustering results.

According to James et al. (2021), feature scaling is a critical preprocessing step in clustering analysis because distance-based algorithms such as K-Means are highly sensitive to differences in variable magnitude. Proper scaling improves model stability and ensures that cluster formation is based on meaningful similarity rather than scale dominance.

## **7.2 Elbow Method**

The Elbow Method is used to determine the optimal number of clusters (K) by evaluating the Within-Cluster Sum of Squares (WCSS), which measures the compactness of data points within each cluster.

In this method, different values of K are tested, and the corresponding WCSS values are plotted. The optimal number of clusters is identified at the point where the rate of decrease in WCSS begins to slow significantly, forming an “elbow-shaped” curve. This point represents a balance between model complexity and clustering efficiency.

From the results of this study, the elbow point is observed where adding additional clusters no longer results in a significant reduction in WCSS. This indicates that the selected number of clusters effectively captures the underlying structure of disaster impact patterns without overfitting.

According to Jain (2010), the Elbow Method is one of the most widely used heuristics for selecting K in K-Means clustering, as it provides a practical balance between interpretability and computational efficiency in unsupervised learning.

The Elbow Method is used to determine the optimal number of clusters (K) by evaluating the Within-Cluster Sum of Squares (WCSS), which measures the compactness of data points within each cluster.

The results show a sharp decrease in WCSS from K=1 to K=3, after which the rate of decrease begins to slow significantly. This point forms a clear “elbow” at K=3, indicating that three clusters provide the most appropriate balance between model simplicity and clustering effectiveness.

Beyond K=3, the reduction in WCSS becomes marginal, suggesting that adding more clusters does not significantly improve the model’s ability to explain variance within the dataset. Therefore, K=3 is selected as the optimal number of clusters for K-Means clustering in this study.

According to Kaufman and Rousseeuw (1990), the Elbow Method is a widely accepted heuristic for determining the optimal number of clusters by identifying diminishing returns in variance reduction. This approach ensures that clustering results are both interpretable and computationally efficient.

## **7.3 K-Means Clustering**

K-Means clustering is applied to group disaster events based on similarities in human and economic impact indicators. The aim is to identify natural severity-based groupings within the dataset.

The silhouette score obtained is 0.9579, indicating an extremely strong level of separation between clusters. This suggests that the data points within each cluster are highly similar to each other, while being clearly distinct from points in other clusters.

Such a high silhouette score demonstrates that the K-Means clustering model is highly effective in identifying well-defined groups within the disaster impact dataset. It confirms that disaster events can be distinctly classified into low, moderate, and high-impact categories based on the selected variables.

Overall, the clustering results are highly reliable and statistically well-separated, supporting the robustness of the model for disaster severity classification and risk segmentation.

## **7.4 Cluster Visualization**

The scatter plot visualises the distribution of disaster events across clusters based on human and economic impact variables.

The K-Means clustering visualization groups disaster events into three distinct clusters based on total affected population and total economic damage.

Cluster 0 represents low-impact disasters and forms the largest and most densely populated group. These observations are concentrated near the lower range of both variables, indicating that most disaster events in the dataset have relatively low human and economic impact.

Cluster 1 represents moderate-impact disasters, which are distributed in the mid-range of both variables. These events reflect transitional severity levels where either human impact or economic loss begins to increase but does not reach extreme levels.

Cluster 2 represents high-impact disasters and is clearly separated from other clusters. These events are characterised by extremely high values in both total affected population and economic damage, indicating the presence of extreme outlier events that significantly influence overall disaster severity.

Although the clusters show general separation, some overlap is present, which is expected in real-world disaster data due to variability in hazard types and reporting differences. This demonstrates that disaster impacts do not form perfectly separable groups but instead exist along a continuous severity spectrum.

Overall, the results confirm that disaster events naturally form structured severity-based groupings. The clustering is driven primarily by differences in magnitude of human exposure and economic loss, highlighting the effectiveness of K-Means clustering in identifying hidden patterns in disaster risk data.

## **7.5 Machine Learning Summary**

The K-Means clustering analysis demonstrates that disaster events can be effectively grouped into distinct severity-based categories using human and economic impact variables. The high silhouette score further confirms that the clusters are well-separated and statistically reliable.

Overall, the machine learning results complement the exploratory data analysis findings by providing a structured segmentation of disaster events, highlighting clear differences in disaster severity levels across the dataset.

## 8.0 Conclusion

This study successfully analysed global disaster patterns between 2020 and 2025 using EM-DAT data and data science techniques, including exploratory data analysis, statistical analysis, and K-Means clustering.

The findings address all research objectives. Flood and storm were identified as the most frequent disaster types, confirming that hydrometeorological hazards dominate global disaster occurrences. Regional analysis showed that disaster distribution is uneven, with Asia and the Americas recording higher concentrations of events over time. Correlation analysis revealed that human impact and economic losses are only weakly correlated, indicating that disaster severity does not follow a single linear relationship. K-Means clustering successfully grouped disaster events into three distinct severity levels: low, moderate, and high-impact disasters.

Overall, the results demonstrate that global disaster impacts are highly heterogeneous across both spatial and temporal dimensions. The combination of exploratory data analysis and machine learning techniques provides a deeper understanding of disaster risk patterns that cannot be captured through descriptive analysis alone.

The study confirms that disaster risk is multidimensional and requires both statistical and machine learning approaches for proper interpretation. The clustering results further highlight that disaster events naturally form distinct severity-based groupings driven by variations in human exposure and economic vulnerability.

In conclusion, this study provides a comprehensive data-driven understanding of global disaster patterns and supports the importance of advanced analytical techniques in improving disaster risk assessment and decision-making.

# **References**

Centre for Research on the Epidemiology of Disasters. (2024). EM-DAT: The international disaster database. Université catholique de Louvain. https://www.emdat.be

Guyon, I., & Elisseeff, A. (2003). An introduction to variable and feature selection. Journal of Machine Learning Research, 3, 1157–1182.

Han, J., Kamber, M., & Pei, J. (2012). Data mining: Concepts and techniques (3rd ed.). Morgan Kaufmann.

Hastie, T., Tibshirani, R., & Friedman, J. (2009). The elements of statistical learning: Data mining, inference, and prediction (2nd ed.). Springer. https://doi.org/10.1007/978-0-387-84858-7

Intergovernmental Panel on Climate Change. (2023). Climate change 2023: Synthesis report. https://www.ipcc.ch/report/ar6/syr/

James, G., Witten, D., Hastie, T., & Tibshirani, R. (2021). An introduction to statistical learning: With applications in R (2nd ed.). Springer. https://www.statlearning.com

Kaufman, L., & Rousseeuw, P. J. (1990). Finding groups in data: An introduction to cluster analysis. Wiley.

Little, R. J. A., & Rubin, D. B. (2019). Statistical analysis with missing data (3rd ed.). Wiley. https://doi.org/10.1002/9781119482260

McKinney, W. (2010). Data structures for statistical computing in Python. In Proceedings of the 9th Python in Science Conference (pp. 56–61).

Rahm, E., & Do, H. H. (2000). Data cleaning: Problems and current approaches. IEEE Data Engineering Bulletin, 23(4), 3–13.

Tukey, J. W. (1977). Exploratory data analysis. Addison-Wesley.

United Nations Office for Disaster Risk Reduction. (2022). Global assessment report on disaster risk reduction. https://www.undrr.org

World Meteorological Organization. (2024). State of the global climate 2024. https://wmo.int